# Bab 13 · Aljabar Linear dan Turunan Numerik

**Notebook praktikum mahasiswa**  
Versi 2.0 · Pemrograman Komputer

- Memeriksa solusi dengan residu dan bilangan kondisi.
- Menyelesaikan kuadrat terkecil tanpa invers eksplisit.
- Memeriksa turunan dan gradien melalui beda pusat.

### Petunjuk menjalankan sel · versi 2.0

Jalankan sel berurutan dari atas ke bawah. Setiap fungsi mandiri diletakkan pada sel tersendiri; sel pemanggilan atau pengujiannya menyusul setelah definisi. Setelah menyunting fungsi, jalankan ulang sel definisinya, lalu sel pengujiannya.

Sel persiapan dan fungsi pemeriksa cukup dijalankan; bagian yang Anda kerjakan ditandai **[ISI KODE]**. Metode yang membentuk satu kelas serta fungsi bersarang tetap disatukan karena merupakan satu kesatuan Python.

## Alur praktikum

**Duga → Jalankan → Selidiki → Isi kode → Periksa → Jelaskan**

Perkiraan waktu: 90–120 menit. Kerjakan berpasangan; tukar peran penulis kode dan pemeriksa setiap dua latihan.

| Penanda | Yang Anda kerjakan |
|---|---|
| [BACA] | Pahami konsep, kontrak fungsi, dan kasus batas. |
| [DUGA] | Tulis prediksi sebelum menjalankan contoh. |
| [COBA] | Jalankan contoh dan ubah satu hal untuk menyelidiki hasilnya. |
| [ISI KODE] | Lengkapi fungsi atau kelas; pertahankan nama dan parameternya. |
| [CEK OTOMATIS] | Jalankan pengujian yang terlihat, lalu gunakan pesannya untuk memperbaiki kode. |
| [REFLEKSI] | Jelaskan alasan dan bukti, bukan hanya menyalin keluaran. |

Impor file `.ipynb` ini ke notebook Python di Kaggle. Gunakan CPU; data kecil disediakan dalam notebook. Jalankan sel dari atas ke bawah. Pustaka yang diperlukan diimpor pada sel persiapan; tidak ada perintah instalasi atau unduhan.

`BELUM DIISI` adalah status normal pada notebook awal. Ganti `raise BelumDiisi()` dengan pekerjaan Anda. `LULUS` berarti memenuhi kasus uji yang tersedia, bukan bukti bahwa semua kemungkinan input sudah benar. Sel pengujian harus tetap utuh.

Jika kode berulang tanpa selesai, hentikan eksekusi, periksa batas perulangan, lalu jalankan ulang. Sebelum mengumpulkan, mulai ulang sesi Python dan jalankan seluruh sel agar hasil tidak bergantung pada variabel lama.

### Identitas

- Nama: …
- NIM: …
- Rekan diskusi: …
- Tanggal: …

**Persiapan dan pengaturan** · bagian 1 dari 9

In [ ]:
# [COBA] Jalankan sekali di awal; pemeriksaan tersedia untuk dibaca.
import math
import sys
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

**Definisi `BelumDiisi`** · bagian 2 dari 9

In [ ]:
class BelumDiisi(Exception):
    """Penanda latihan yang belum dikerjakan."""

**Definisi `sama`** · bagian 3 dari 9

In [ ]:
def sama(aktual, harapan):
    assert aktual == harapan, f"Diharapkan {harapan!r}; diperoleh {aktual!r}"

**Definisi `dekat`** · bagian 4 dari 9

In [ ]:
def dekat(aktual, harapan, atol=1e-8, rtol=1e-7):
    assert math.isclose(
        aktual, harapan, abs_tol=atol, rel_tol=rtol
    ), f"Diharapkan sekitar {harapan!r}; diperoleh {aktual!r}"

**Definisi `harus_galat`** · bagian 5 dari 9

In [ ]:
def harus_galat(jenis, panggil):
    try:
        panggil()
    except BelumDiisi:
        raise
    except jenis:
        return
    raise AssertionError(f"Seharusnya memunculkan {jenis.__name__}")

**Persiapan dan pengaturan** · bagian 6 dari 9

In [ ]:
DAFTAR_UJI = {}

**Definisi `cek`** · bagian 7 dari 9

In [ ]:
def cek(nomor, fungsi_uji, tampil=True):
    DAFTAR_UJI[nomor] = fungsi_uji
    try:
        fungsi_uji()
        status, pesan = "LULUS", "Semua kasus uji pada latihan ini sesuai."
    except BelumDiisi:
        status, pesan = (
            "BELUM DIISI",
            "Lengkapi sel [ISI KODE], jalankan, lalu ulangi pemeriksaan.",
        )
    except AssertionError as err:
        status, pesan = (
            "PERLU PERBAIKAN",
            str(err) or "Hasil belum sesuai kontrak latihan.",
        )
    except Exception as err:
        status, pesan = "GALAT", f"{type(err).__name__}: {err}"
    if tampil:
        print(f"Latihan {nomor} | {status}\n{pesan}")
    return status

**Definisi `rekap`** · bagian 8 dari 9

In [ ]:
def rekap():
    # Uji ulang fungsi terkini agar rekap tidak memakai status lama.
    hasil = {
        nomor: cek(nomor, uji, tampil=False)
        for nomor, uji in sorted(DAFTAR_UJI.items())
    }
    for nomor, status in hasil.items():
        print(f"  Latihan {nomor}: {status}")
    lulus = sum(s == "LULUS" for s in hasil.values())
    print(f"\nKemajuan uji otomatis: {lulus}/{JUMLAH_LATIHAN} latihan lulus.")
    print(
        "Refleksi, penjelasan, dan kualitas penyajian diperiksa bersama asisten."
    )
    return hasil

**Persiapan dan pengaturan** · bagian 9 dari 9

In [ ]:
print("Python:", sys.version.split()[0])
print("Siap. Jalankan notebook dari atas ke bawah.")
JUMLAH_LATIHAN = 4
import numpy as np

print("NumPy:", np.__version__)

## [BACA] Konsep inti

Residu kecil berarti persamaan hampir terpenuhi, tetapi belum menjamin solusi dekat nilai sebenarnya ketika matriks berkondisi buruk. Gunakan solve atau lstsq sesuai masalah. Beda pusat memakai f(x+h) dan f(x−h); langkah terlalu kecil dapat memperbesar pengaruh pembulatan. Saat mengganggu satu komponen vektor, gunakan salinan agar input tidak berubah.

## [DUGA] Prediksi sebelum eksekusi

Apakah residu yang sangat kecil selalu berarti galat solusi juga sangat kecil?

**Prediksi saya:** …

**Alasan:** …

In [ ]:
# [COBA]
for n in [4, 10]:
    k = np.arange(n)
    H = 1 / (k[:, None] + k[None, :] + 1)
    benar = np.ones(n)
    b = H @ benar
    hasil = np.linalg.solve(H, b)
    print(
        n,
        "kondisi:",
        np.linalg.cond(H),
        "residu:",
        np.linalg.norm(H @ hasil - b),
        "galat solusi:",
        np.linalg.norm(hasil - benar),
    )

**[REFLEKSI]** Apa perbedaan prediksi dan hasil? Ubah satu input pada contoh, tulis hasilnya, lalu jelaskan konsep yang ditunjukkan.

**Jawaban:** …

## Latihan 1 · Solusi dan residu

**[ISI KODE]**

Buat `selesaikan(A, b)` untuk matriks float persegi nonsingular dan vektor b. Kembalikan `(x, norma_residu)` dengan norma Euclidean ||Ax−b||. Gunakan np.linalg.solve; biarkan LinAlgError diteruskan untuk matriks singular. Input tidak berubah.

> Petunjuk: Kalikan kembali A dan solusi untuk menghitung residu.

In [ ]:
# [ISI KODE]
def selesaikan(A, b):
    raise BelumDiisi()

**Definisi `uji_01`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_01():
    A = np.array([[2.0, 1.0], [1.0, 3.0]])
    b = np.array([5.0, 5.0])
    awal = A.copy()
    x, r = selesaikan(A, b)
    np.testing.assert_allclose(x, [2, 1])
    dekat(r, 0, atol=1e-12)
    np.testing.assert_array_equal(A, awal)
    harus_galat(
        np.linalg.LinAlgError, lambda: selesaikan(np.ones((2, 2)), np.ones(2))
    )

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(1, uji_01)

## Latihan 2 · Kuadrat terkecil

**[ISI KODE]**

Buat `fit_kuadrat_terkecil(X, y)` yang mengembalikan vektor bobot dari np.linalg.lstsq dengan rcond=None. X sudah memuat kolom bias bila diperlukan. Dukung kolom bergantung linear; jangan gunakan invers X.T @ X.

> Petunjuk: lstsq mengembalikan beberapa hasil; ambil komponen bobotnya.

In [ ]:
# [ISI KODE]
def fit_kuadrat_terkecil(X, y):
    raise BelumDiisi()

**Definisi `uji_02`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_02():
    X = np.column_stack([np.ones(4), np.arange(4)])
    y = np.array([1.0, 3.0, 5.0, 7.0])
    np.testing.assert_allclose(fit_kuadrat_terkecil(X, y), [1, 2], atol=1e-12)
    X_rank = np.ones((3, 2))
    w = fit_kuadrat_terkecil(X_rank, np.array([2.0, 2.0, 2.0]))
    np.testing.assert_allclose(w, [1, 1], atol=1e-12)
    X_noisy = np.column_stack([np.ones(3), [-1, 0, 1]])
    np.testing.assert_allclose(
        fit_kuadrat_terkecil(X_noisy, np.array([0.0, 2.0, 3.0])), [5 / 3, 1.5]
    )

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(2, uji_02)

## Latihan 3 · Turunan beda pusat

**[ISI KODE]**

Buat `beda_pusat(f, x, h)` untuk h > 0, dengan rumus [f(x+h)−f(x−h)]/(2h). Tolak h ≤ 0 dengan ValueError. Fungsi f mengembalikan skalar.

> Petunjuk: Pembagi mencakup jarak antara dua titik, yaitu 2h.

In [ ]:
# [ISI KODE]
def beda_pusat(f, x, h):
    raise BelumDiisi()

**Definisi `uji_03`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_03():
    dekat(beda_pusat(lambda x: x * x, 3, 1e-4), 6, atol=1e-8)
    dekat(beda_pusat(np.sin, 1, 1e-5), np.cos(1), atol=1e-8)
    dekat(beda_pusat(lambda x: 7, 2, 0.01), 0)
    harus_galat(ValueError, lambda: beda_pusat(np.sin, 1, 0))
    harus_galat(ValueError, lambda: beda_pusat(np.sin, 1, -0.1))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(3, uji_03)

## Latihan 4 · Pemeriksa gradien numerik

**[ISI KODE]**

Buat `gradien_numerik(f, w, h=1e-5)` untuk vektor float w. Gunakan beda pusat pada satu komponen setiap langkah; kembalikan ndarray dengan bentuk sama. Input w harus tetap utuh. Tolak h ≤ 0 dengan ValueError.

> Petunjuk: Setiap komponen memerlukan dua salinan w, bukan dua nama yang merujuk objek sama.

In [ ]:
# [ISI KODE]
def gradien_numerik(f, w, h=1e-5):
    raise BelumDiisi()

**Definisi `uji_04`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_04():
    w = np.array([1.0, 2.0])
    awal = w.copy()
    g = gradien_numerik(lambda v: v[0] ** 2 + 3 * v[0] * v[1] + v[1] ** 2, w)
    np.testing.assert_allclose(g, [8, 7], atol=1e-7)
    np.testing.assert_array_equal(w, awal)
    np.testing.assert_allclose(
        gradien_numerik(lambda v: float(v @ v), np.array([-2.0, 0.0, 3.0])),
        [-4, 0, 6],
        atol=1e-7,
    )
    harus_galat(ValueError, lambda: gradien_numerik(lambda v: v @ v, w, 0))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(4, uji_04)

## [REFLEKSI] Refleksi akhir

Bagian ini membantu Anda merangkum pemahaman, mengenali kesulitan, dan menjelaskan alasan di balik kode. Tulis jawaban singkat berdasarkan percobaan Anda; bukan sekadar menyalin keluaran.

1. Pilih satu latihan. Jelaskan alur kode Anda dengan satu contoh input dan hasilnya.
2. Tuliskan satu kesalahan yang sempat terjadi, penyebabnya, dan cara memperbaikinya.
3. Usulkan satu kasus uji tambahan yang belum tercakup. Nyatakan hasil yang Anda harapkan dan alasannya.
4. Apa batas kesimpulan yang boleh dibuat dari hasil praktikum ini?

**Jawaban:** …

### Tantangan pengembangan

Tambahkan kasus uji usulan Anda pada sel di bawah. Pastikan kasus tersebut bisa membedakan implementasi benar dan satu kesalahan yang masuk akal. Diskusikan dengan asisten sebelum mengubah kontrak fungsi.

In [ ]:
# [ISI KODE OPSIONAL] Tambahkan eksperimen atau pengujian buatan Anda.
# Jelaskan harapan Anda pada komentar sebelum menjalankannya.

In [ ]:
# [CEK OTOMATIS] Uji ulang seluruh latihan yang sudah didaftarkan.
status_akhir = rekap()

## Sebelum mengumpulkan

- [ ] Identitas dan prediksi sudah diisi.
- [ ] Semua latihan sudah dikerjakan dan diperiksa dari sesi baru.
- [ ] refleksi akhir berisi penjelasan dengan bukti keluaran.
- [ ] Notebook disimpan dengan nama dan NIM; jangan hanya mengumpulkan HTML.

Rubrik diskusi: ketepatan kode 60%, penjelasan dan kasus batas 25%, keterbacaan serta kemampuan dijalankan ulang 15%. Rekap otomatis membantu belajar; penilaian akhir tetap memerlukan pemeriksaan asisten.

Rujukan: bab yang bersesuaian pada buku *Python untuk Machine Learning dan Data Science* dan modul praktikum. Latihan di notebook ini merupakan adaptasi terarah untuk praktikum, bukan seluruh soal akhir bab.